# धडा 18 (पुढील): पावत्या जे सिद्ध करतात की एक *मनुष्य* क्रियेचे प्राधिकरण दिले

हा धडा काय काय **एजंट** ने केले आणि काय **गेट** ने ठरवले ते सिद्ध करतो. हा नोटबुक गायब अर्धा भाग जोडतो: पुरावा की एक **नावीन मनुष्याने** **निश्चित** क्रियेची मंजुरी दिली — संपूर्ण कॅनॉनिकल क्रियेवर स्वतंत्र, मनुष्याकडे असलेली स्वाक्षरी, ऑफलाइन पडताळलेली.

येथे दोन्ही वस्तू वापरतात **धड्याच्या पावत्यांसारख्याच साचा**: `type` क्षेत्र असलेले थेट पेलोड, जे Ed25519 ने कॅनॉनिकल JCS बायट्सवर सही केलेले आहे, ज्याच्यापासून एक संरचित `signature` ऑब्जेक्ट संलग्न (आणि सही केलेल्या बाइट्समधून वगळलेले) आहे. मंजुरीची पावती एक नवीन `type` (`human.approval.v1`) आहे जी क्रिया प्रकाराबरोबर आहे, त्यामुळे एक `verify_chain` दोन्ही वस्तू प्रकारांसाठी मुख्य नोटबुकमध्ये तयार केलेल्या सारख्या कोडमार्गाने पुरेसे आहे. ही मानवी-मंजुरीची पावती येथे परिभाषित शैक्षणिक संघटन आहे, draft-farley-acta-signed-receipts ने परिभाषित केलेली नाही.

मुख्य नोटबुकमधील डेमो पडताळणी करणाऱ्यापेक्षा एक जाणूनबुजून सुधारणा: येथे पडताळणी करणारा `signature.key_id` ची पडताळणी **पिन केलेल्या की रजिस्ट्रेटीशी** करतो, पावतीमध्ये असलेल्या सार्वजनिक कीवर विश्वास ठेवत नाही. हा उत्पादनाचा व्यवहार आहे, जो धड्याच्या स्वतःच्या चेकलिस्टने शिफारस केला आहे ("पडताळणी सार्वजनिक की प्रकाशित करा"), आणि ज्यामुळे बनावट नाकारली जाते, आणखी स्वतःची की आणण्याचा मार्ग बंद होतो.

या नोटबुकचा नियम: **स्वाक्षरी केलेली मंजुरी स्वतःमध्ये प्राधिकरण नाही.** प्राधिकरण फक्त तेव्हाच असते जेव्हा मंजुरीची पावती आणि क्रिया पावती अजूनही अंमलबजावणीवेळी त्याच कॅनॉनिकल क्रियेशी बांधली जातात, एखाद्या धोरण आवृत्ती, की आणि कालबाह्य होण्याच्या स्थितीच्या अंतर्गत जे अजूनही चालू आहेत, आणि जी मंजुरी अजून वापरलेली नाही. प्रत्येक अपयश एक **विशिष्ट कारणाने** नाकारले जाते, त्यामुळे तुम्हाला *प्राधिकरण जुने झाले* वगळता *अंमलात आणलेली क्रिया बदलली* हा फरक कळतो.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## अचूक क्रिया

मान्यता युनिट म्हणजे **कॅनॉनिकल क्रिया ऑब्जेक्ट** — "रिफंड मंजूर करा" सारखा अस्पष्ट लेबल नाही, तर अचूक, पूर्णपणे निर्दिष्ट क्रिया. संपूर्ण ऑब्जेक्टवर सही करणे (आणि त्यातून डाइजेस्ट तयार करणे) हेच वापरून आम्हाला नंतर पुरावा देता येतो की माणसाने *ही* क्रिया मंजूर केली आहे आणि इतर काही नाही.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## एक लिफाफा, दोन अधिकृत प्राधिकरणे

प्रत्येक पावती हा धड्याचा लिफाफा आहे: एक सपाट पेलोड ज्यात `type` फील्ड असते, तसेच `signature` ऑब्जेक्ट (`alg`, `sig`, `key_id`) जे स्वाक्षरी केलेल्या बाइट्सचा भाग नाही. `verify_envelope` हा दोन्ही पावती प्रकारांसाठी सामायिक संरचनात्मक + स्वाक्षरी तपासणी आहे; जे **पिन केलेले की रजिस्ट्री** `signature.key_id` सोबत जुळवते त्यावरून प्राधिकरणे विभक्त राहतात:

- **मंजूरी पावती** (`human.approval.v1`) — नाव दिलेला मंजूर करणारा, पूर्ण कॅनॉनिकल कृती **आणि त्याचा डाइजेस्ट**, `policy_version`, जारीकरण + कालबाह्यता कालमर्यादा. एकदाच वापर चेन स्तरावर ट्रॅक केला जातो.
- **कृती पावती** (`agent.action.v1`) — एजंट आयडेंटिटी, `run_id`, तीच कॅनॉनिकल कृती **डाइजेस्ट**, कार्यवाहीचा निकाल + टाइमस्टँप, आणि `parent_approval_ref`: मंजुरीची `receipt_hash`, धड्याच्या चेनमधील `previous_receipt_hash` प्रमाणेच कन्व्हेन्शन.

सामायिक `action_digest` फील्ड हा जॉइन आहे ज्यावर बाइंडिंग अवलंबून आहे. `key_id` फक्त शोध सहाय्य म्हणून स्वाक्षरी ऑब्जेक्टमध्ये असते: त्याला वेगळ्या पिन केलेल्या कीकडे वळवल्यास स्वाक्षरी तपासणी अयशस्वी होते, त्यामुळे यात काहीही मूल्य नसते.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: जिथे बाँडिंग प्रत्यक्षात ठरवली जाते

`verify_chain` हा दोन स्वाक्षऱ्यांच्या तपासण्यांवर असलेला सोयीसाठीचा आवरण नाही. हे एकच ठिकाण आहे जिथे सामायिक प्रामाणिक `action_digest`, धोरण/की/कालबाह्यत्व **ताजेपणा** आणि मंजुरीचा **एकदा वापर होण्याचा** तपास एकत्र केला जातो, अत्ताच चालवल्या जाणाऱ्या क्रियाविरोधात.

प्रत्येक अपयश वेगळ्या **कारणासह** नाकारले जाते, जेणेकरून नाकारण्याचा वाचक ठरवू शकतो की अधिकार कालबाह्य झाला आहे का (धोरण बदलले, की फिरवली गेली, मंजुरी कालबाह्य झाली, मंजुरी वापरली गेली) किंवा क्रियान्वित क्रिया अजूनही वैध मंजुरी असतानाही बदलली आहे का (डायजेस्ट बदल). 


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## बाइंडिंग काय पकडते

खालील प्रत्येक प्रकरण एका **अलग कारणास्तव** **बंद** होते. पहिला ब्लॉक हा पारंपरिक संच आहे (छेडछाड, गोंधळलेला डिप्युटी, पुनरावृत्ती, प्राधिकरणावर फसवणूक, खराब इनपुट). दुसरा ब्लॉक तो जो मालमत्तेला खरे बनवतो, केवळ दावा केला जात नाही:

- **जुनी प्राधिकरण** — स्वाक्षरी अजून वैध आहे, परंतु धोरण आवृत्ती बदलली आहे, मंजूरकर्ता की पिन केलेल्या रजिसट्रीमधून फिरवण्यात आली आहे, अथवा अंमलबजावणीपूर्वी मंजुरीची समाप्ती झाली आहे;
- **डाइजेस्ट बदल** — वैध स्वरूपात स्वाक्षरी केलेला क्रिया पावतीचा जिना `parent_approval_ref` खऱ्या मंजुरीकडे निर्देश करतो, परंतु त्या मंजुरीचा प्रमाणित क्रिया डाइजेस्ट वास्तवात चालवल्या जाणाऱ्या क्रियेशी जुळत नाही.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## हे काय सिद्ध करते — आणि काय नाही

**सिद्ध करते:** एका नाविन्यपूर्ण माणसाने *हे नेमके कायनॉनिकल क्रियाकलाप* (पूर्ण क्रिया + डाइजेस्ट, पिन केलेल्या नोंदणीहून सोडवलेल्या कीने स्वाक्षरी केलेले) मान्य केले आहे, आणि एजंटने *अगदी तीच मान्य केलेली क्रिया* (तेच डाइजेस्ट, `receipt_hash` द्वारा मान्यतेशी बांधलेली पावती, धड्याची स्वतःची साखळी संहिता) अंमलात आणली आहे — जेव्हापर्यंत मान्यतेची धोरण आवृत्ती, की, आणि समाप्ती अद्ययावत होती, अगदी एकदाच. जर कोणताही भाग बदलला, तर साखळी बंद होते, आणि नाकारण्याचा कारण तुम्हाला **का** गुणधर्म तुटले ते सांगते: जुनी अधिकार vs. बदललेली क्रिया.

**सिद्ध करत नाही:** की मान्यता UI ने माणसाला ते काय स्वाक्षरी करत आहेत हे दाखवले (WYSIWYS हा स्वतःचा प्रश्न आहे), की की गुंडाळली किंवा चोरी झाली नाही फिरवण्यापूर्वी, किंवा की खालील परिणाम क्रियेच्या अनुरूप होते. स्वाक्षरी केलेले ≠ अधिकृत: जुनी धोरण, फिरवलेली की, कालबाह्य विंडो, किंवा वेगळा डाइजेस्ट येथे काहीही देत नाही.

दोन पावती प्रकार धड्याचा लिफाफा आणि एक `verify_chain` कोड मार्ग हेतुपुरस्सर सामायिक करतात: मुख्य नोटबुकमधील क्रिया पावतींसाठी तुम्ही तयार केलेली बांधणी हीच माणसाच्या मान्यतेची तपासणी करणारी कोड आहे. एक पडताळणी करार, वेगळे पिन केलेले प्राधिकरण, कायनॉनिकल क्रिया डाइजेस्ट आणि आणखी काही नाही यांनी जोडलेले.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
